In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
import os
import sys
print(os.path.join(os.getcwd(),"..", ".."))
path = os.path.join(os.getcwd(),"..", "..")
print(sys.path)
sys.path.append(path)

In [0]:
from utils.transformations import reusable

##DimUser

In [0]:
df_user = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format", "parquet")\
            .option("cloudFiles.schemaLocation", "abfss://silver@spotifyprojectvyom.dfs.core.windows.net/DimUser/checkpoints")\
            .load("abfss://bronze@spotifyprojectvyom.dfs.core.windows.net/DimUser/")

In [0]:
df_user = df_user.withColumn("user_name", upper(col("user_name")))

In [0]:
df_user_obj = reusable()
df_user = df_user_obj.dropColumns(df_user, '_rescued_data')
df_user = df_user.dropDuplicates(['user_id'])

In [0]:
display(df_user)

In [0]:
df_user.writeStream.format("delta").outputMode("append")\
        .option("checkpointLocation", "abfss://silver@spotifyprojectvyom.dfs.core.windows.net/DimUser/checkpoints")\
        .trigger(once=True)\
        .option("path", "abfss://silver@spotifyprojectvyom.dfs.core.windows.net/DimUser/data/")\
        .toTable("spotify_data.silver.DimUser")

##DimArtist

In [0]:
df_artist = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemalocation", "abfss://silver@spotifyprojectvyom.dfs.core.windows.net/DimArtist/checkpoints")\
                .load("abfss://bronze@spotifyprojectvyom.dfs.core.windows.net/DimArtist/")

In [0]:
df_artist = df_user_obj.dropColumns(df_artist, '_rescued_data')
df_artist = df_artist.dropDuplicates(["artist_id"])

In [0]:
display(df_artist)

In [0]:
df_artist.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation", "abfss://silver@spotifyprojectvyom.dfs.core.windows.net/DimArtist/checkpoints")\
        .trigger(once=True)\
        .option("path", "abfss://silver@spotifyprojectvyom.dfs.core.windows.net/DimArtist/data/")\
        .toTable("spotify_data.silver.DimArtist")

## DimTracks

In [0]:
df_tracks = spark.readStream.format('cloudFiles')\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation", "abfss://silver@spotifyprojectvyom.dfs.core.windows.net/DimTrack/checkpoints")\
                .load("abfss://bronze@spotifyprojectvyom.dfs.core.windows.net/DimTrack/")

In [0]:
display(df_tracks)

In [0]:
df_tracks = df_user_obj.dropColumns(df_tracks, '_rescued_data')
df_tracks = df_tracks.withColumn("track_flag", when(col("duration_sec") >= 250, "High")\
                                                .when(col("duration_sec")>=150, "Medium")\
                                                .otherwise("Low"))
df_tracks = df_tracks.withColumn("track_name",regexp_replace(col('track_name'),'-',' '))


In [0]:
df_tracks.writeStream.format("delta")\
                .outputMode("append")\
                .option("checkpointLocation", "abfss://silver@spotifyprojectvyom.dfs.core.windows.net/DimTrack/checkpoints")\
                .trigger(once = True)\
                .option("path", "abfss://silver@spotifyprojectvyom.dfs.core.windows.net/DimTrack/data/")\
                .toTable("spotify_data.silver.DimTrack")

##DimDate

In [0]:
df_date = spark.readStream.format('cloudFiles')\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation", "abfss://silver@spotifyprojectvyom.dfs.core.windows.net/DimDate/checkpoints")\
                .load("abfss://bronze@spotifyprojectvyom.dfs.core.windows.net/DimDate/")

In [0]:
df_date = df_user_obj.dropColumns(df_date, '_rescued_data')


In [0]:
display(df_date)

In [0]:
df_date.writeStream.format("delta")\
                .outputMode("append")\
                .option("checkpointLocation", "abfss://silver@spotifyprojectvyom.dfs.core.windows.net/DimDate/checkpoints")\
                .trigger(once = True)\
                .option("path", "abfss://silver@spotifyprojectvyom.dfs.core.windows.net/DimDate/data/")\
                .toTable("spotify_data.silver.DimDate")

##FactStream

In [0]:
df_fact = spark.readStream.format('cloudFiles')\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation", "abfss://silver@spotifyprojectvyom.dfs.core.windows.net/FactStream/checkpoints")\
                .load("abfss://bronze@spotifyprojectvyom.dfs.core.windows.net/FactStream/")

In [0]:
df_fact = df_user_obj.dropColumns(df_fact, '_rescued_data')

In [0]:
df_fact.writeStream.format("delta")\
                .outputMode("append")\
                .option("checkpointLocation", "abfss://silver@spotifyprojectvyom.dfs.core.windows.net/FactStream/checkpoints")\
                .trigger(once = True)\
                .option("path", "abfss://silver@spotifyprojectvyom.dfs.core.windows.net/FactStream/data/")\
                .toTable("spotify_data.silver.FactStream")


##OPTIMIZE FILES

In [0]:
%sql
OPTIMIZE spotify_data.silver.DimUser;
OPTIMIZE spotify_data.silver.DimTrack;
OPTIMIZE spotify_data.silver.DimArtist;
OPTIMIZE spotify_data.silver.DimDate;

In [0]:
%sql
DESCRIBE HISTORY spotify_data.silver.DimUser;
DESCRIBE HISTORY spotify_data.silver.DimArtist;